In [ ]:
import numpy as np
import tensorflow as tf
from tensorflow.keras.layers import Embedding, Dense

tf.random.set_seed(42)

# Sample vocabulary
words = [
    "good", "bad", "excellent", "poor",
    "movie", "acting", "very", "not"
]

word_to_id = {word: i for i, word in enumerate(words)}

# Sentence trees and sentiment labels
data = [
    (("good", "movie"), 1),
    (("bad", "movie"), 0),
    (("excellent", "acting"), 1),
    (("poor", "acting"), 0),
    (("very", ("good", "movie")), 1),
    (("not", ("good", "movie")), 0),
    (("very", ("bad", "movie")), 0),
    (("not", ("bad", "movie")), 1)
]

# Recursive Neural Network
class RecursiveNN(tf.keras.Model):
    def __init__(self):
        super().__init__()
        self.embedding = Embedding(len(words), 16)
        self.compose = Dense(16, activation="tanh")
        self.output_layer = Dense(1, activation="sigmoid")

    def build_tree(self, tree):
        if isinstance(tree, str):
            word_id = word_to_id[tree]
            return self.embedding(
                tf.constant([word_id])
            )[0]

        left = self.build_tree(tree[0])
        right = self.build_tree(tree[1])

        combined = tf.concat([left, right], axis=0)
        return self.compose(combined)

    def call(self, tree):
        root = self.build_tree(tree)
        return self.output_layer(root)

model = RecursiveNN()
optimizer = tf.keras.optimizers.Adam(0.01)
loss_fn = tf.keras.losses.BinaryCrossentropy()

# Training
for epoch in range(100):
    total_loss = 0

    for tree, label in data:
        with tf.GradientTape() as tape:
            prediction = model(tree)
            loss = loss_fn(
                tf.constant([[float(label)]]),
                tf.reshape(prediction, (1, 1))
            )

        gradients = tape.gradient(
            loss, model.trainable_variables
        )
        optimizer.apply_gradients(
            zip(gradients, model.trainable_variables)
        )

        total_loss += loss.numpy()

    if (epoch + 1) % 20 == 0:
        print("Epoch:", epoch + 1,
              "Loss:", round(total_loss, 4))

# Test predictions
print("\nSentiment Predictions:")

for tree, label in data:
    prediction = model(tree).numpy()[0]
    sentiment = "Positive" if prediction >= 0.5 else "Negative"

    print("Sentence Tree:", tree)
    print("Predicted Sentiment:", sentiment)
    print("----------------------")